In [7]:
# Model training for credit card fraud detection
import os
import json
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import argparse
from azureml.core import Dataset, Workspace, Datastore
from azureml.data.datapath import DataPath

parser = argparse.ArgumentParser()
parser.add_argument("--input_data", type=str, default="./data")
parser.add_argument("--output_model", type=str, default="./outputs")

args, _ = parser.parse_known_args()

INPUT_DIR = args.input_data
OUTPUT_DIR = args.output_model

print("Input:", INPUT_DIR)
print("Output:", OUTPUT_DIR)

# Get Azure ML workspace
ws = Workspace.from_config()

# -----------------------------
# Load data
# -----------------------------

# Reference the datastore and path
datastore = Datastore.get(ws, 'workspaceblobstore')
datastore_path = 'UI/2026-04-04_090211_UTC/creditcard_train.csv'

# Use DataPath to reference the file in the datastore
dataset = Dataset.File.from_files(path=DataPath(datastore, datastore_path))
downloaded_paths = dataset.download(target_path=INPUT_DIR, overwrite=True)
data_path = downloaded_paths[0]

data = pd.read_csv(data_path)

X = data.drop("Class", axis=1)
y = data["Class"]

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------
# Train model
# -----------------------------
model = RandomForestClassifier(random_state=42)
model.fit(x_train, y_train)

# -----------------------------
# Evaluate
# -----------------------------
y_pred = model.predict(x_test)

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1_score": f1_score(y_test, y_pred)
}

print("📊 Metrics:", metrics)

# -----------------------------
# Save model artifacts
# -----------------------------
os.makedirs(OUTPUT_DIR, exist_ok=True)

joblib.dump(model, os.path.join(OUTPUT_DIR, "model.pkl"))

with open(os.path.join(OUTPUT_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f)

print("✅ Model and metrics saved locally")

# -----------------------------
# Upload outputs to blob storage
# -----------------------------
datastore.upload(src_dir=OUTPUT_DIR, target_path='outputs', overwrite=True, show_progress=False)

print("✅ Outputs uploaded to blob storage")

Input: ./data
Output: ./outputs
{'infer_column_types': 'False', 'activity': 'download'}
{'infer_column_types': 'False', 'activity': 'download', 'activityApp': 'FileDataset'}
📊 Metrics: {'accuracy': 0.9994333333333333, 'precision': 1.0, 'recall': 0.746268656716418, 'f1_score': 0.8547008547008547}
✅ Model and metrics saved locally
✅ Outputs uploaded to blob storage
